# Store Sales Prediction — EDA Visualization
========================================
**Project**: Kaggle Corporación Favorita Store Sales Prediction

**Goal**: Gain deep understanding of data patterns through visualization

**Data Source**: `../data/processed/` (cleaned) or `../data/raw/` (original)

In [2]:
# Cell 1: Import Libraries
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings("ignore")

# Plot style settings
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 150
sns.set_style("whitegrid")
sns.set_palette("tab10")

print("Libraries imported successfully.")

Libraries imported successfully.


In [ ]:
# Cell 2: Load Data (prefer cleaned data, fall back to raw)
BASE_DIR = os.path.abspath("")
PROCESSED_DIR = os.path.join(BASE_DIR, "..", "data", "processed")
RAW_DIR = os.path.join(BASE_DIR, "..", "data", "raw")

def load_data(filename, parse_dates=None):
    """Load from processed/ if available, otherwise fall back to raw/."""
    processed_path = os.path.join(PROCESSED_DIR, filename)
    raw_path = os.path.join(RAW_DIR, filename)
    if os.path.exists(processed_path):
        print(f"[processed] Loading {filename}")
        return pd.read_csv(processed_path, parse_dates=parse_dates)
    else:
        print(f"[raw]      Loading {filename}")
        return pd.read_csv(raw_path, parse_dates=parse_dates)

df_train = load_data("train_cleaned.csv", parse_dates=["date"])
df_stores = load_data("stores_cleaned.csv")
df_oil = load_data("oil_cleaned.csv", parse_dates=["date"])
df_holidays = load_data("holidays_cleaned.csv", parse_dates=["date"])
df_transactions = load_data("transactions_cleaned.csv", parse_dates=["date"])

print(f"\ntrain:         {df_train.shape[0]:,} rows")
print(f"stores:        {df_stores.shape[0]} rows")
print(f"oil:           {df_oil.shape[0]} rows")
print(f"holidays:      {df_holidays.shape[0]} rows")
print(f"transactions:  {df_transactions.shape[0]:,} rows")

In [ ]:
# Cell 3: 【图表1】总销量时间趋势折线图 (2013-2017)
daily_total_sales = df_train.groupby("date")["sales"].sum().reset_index()

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(daily_total_sales["date"], daily_total_sales["sales"],
        linewidth=0.5, color="#1f77b4", alpha=0.85)
ax.set_title("Daily Total Sales (2013-2017)", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Total Sales")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))

# 标注地震事件
earthquake_date = pd.Timestamp("2016-04-16")
ax.axvline(earthquake_date, color="red", linestyle="--", alpha=0.6, linewidth=1)
ax.annotate("2016 Manabi Earthquake", xy=(earthquake_date, daily_total_sales["sales"].max()),
            xytext=(15, 0), textcoords="offset points", fontsize=9, color="red",
            rotation=90, va="top")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 4: 【图表2】各品类销量占比
family_sales = df_train.groupby("family")["sales"].sum().sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# 横向柱状图
colors = sns.color_palette("vlag_r", len(family_sales))
axes[0].barh(family_sales.index, family_sales.values, color=colors)
axes[0].set_title("Total Sales by Product Family", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Total Sales")
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))

# 饼图（Top 8 + Others）
top8 = family_sales.nlargest(8)
others = family_sales.sum() - top8.sum()
pie_data = pd.concat([top8, pd.Series({"OTHERS": others})])
wedges, texts, autotexts = axes[1].pie(
    pie_data.values, labels=pie_data.index, autopct='%1.1f%%',
    colors=sns.color_palette("Set2", len(pie_data)),
    startangle=140, pctdistance=0.85
)
for at in autotexts:
    at.set_fontsize(7)
axes[1].set_title("Sales Share: Top 8 Families + Others", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
# Cell 5: 【图表3】各商店类型销量分布 — 箱线图
# 计算每个 (store_nbr, type) 的总销量用于对比
store_type_sales = df_train.groupby(["store_nbr", "type"])["sales"].sum().reset_index()
type_order = sorted(store_type_sales["type"].unique())

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 箱线图
sns.boxplot(
    x="type", y="sales", data=store_type_sales,
    order=type_order, palette="Set2", ax=axes[0],
    width=0.5
)
axes[0].set_title("Sales Distribution by Store Type", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Store Type")
axes[0].set_ylabel("Total Sales per Store")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))

# 每个 type 的商店数量
type_counts = df_stores["type"].value_counts().reindex(type_order)
bars = axes[1].bar(type_counts.index, type_counts.values, color=sns.color_palette("Set2", len(type_counts)))
for bar, val in zip(bars, type_counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 str(val), ha='center', fontsize=11, fontweight='bold')
axes[1].set_title("Number of Stores per Type", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Store Type")
axes[1].set_ylabel("Count")
axes[1].set_ylim(0, max(type_counts.values) * 1.2)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 6: 【图表4】星期几销量模式
df_train_copy = df_train.copy()
df_train_copy["dayofweek"] = df_train_copy["date"].dt.dayofweek
df_train_copy["dow_name"] = df_train_copy["date"].dt.day_name()

dow_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 每日总销量
dow_sales = df_train_copy.groupby("dow_name")["sales"].sum().reindex(dow_order)
colors = sns.color_palette("Blues_d", 7)
axes[0].bar(dow_sales.index, dow_sales.values, color=colors, edgecolor="white")
axes[0].set_title("Total Sales by Day of Week", fontsize=13, fontweight="bold")
axes[0].set_xlabel("")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))
axes[0].tick_params(axis='x', rotation=45)

# 按商店类型分组的星期几模式
dow_type = df_train_copy.groupby(["type", "dow_name"])["sales"].mean().reset_index()
dow_pivot = dow_type.pivot(index="dow_name", columns="type", values="sales").reindex(dow_order)
dow_pivot_norm = dow_pivot.div(dow_pivot.max(axis=1), axis=0)  # 归一化
sns.heatmap(dow_pivot_norm.T, cmap="YlOrRd", annot=dow_pivot.round(0).T.astype(int),
            fmt='d', linewidths=1, ax=axes[1], cbar_kws={'label': 'Normalized'})
axes[1].set_title("Avg Sales: Day of Week × Store Type", fontsize=13, fontweight="bold")
axes[1].set_xlabel("")
axes[1].set_ylabel("Store Type")

plt.tight_layout()
plt.show()

In [ ]:
# Cell 7: 【图表5】月度销量热力图 — Month × Year
df_train_copy = df_train.copy()
df_train_copy["year"] = df_train_copy["date"].dt.year
df_train_copy["month"] = df_train_copy["date"].dt.month

monthly_sales = df_train_copy.groupby(["year", "month"])["sales"].sum().reset_index()
monthly_pivot = monthly_sales.pivot(index="month", columns="year", values="sales")

month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
monthly_pivot.index = month_names[:len(monthly_pivot)]

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(
    monthly_pivot / 1e6,  # 转为百万单位
    annot=True, fmt='.1f', cmap="YlOrRd",
    linewidths=1.5, cbar_kws={'label': 'Total Sales (Millions)'},
    ax=ax, annot_kws={'fontsize': 9}
)
ax.set_title("Monthly Total Sales Heatmap (Millions)", fontsize=14, fontweight="bold")
ax.set_ylabel("Month")
ax.set_xlabel("Year")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 8: 【图表6】节假日效应对比 — 假日 vs 非假日销量
# 筛选全国性 Holiday 日期
national_holidays = df_holidays[
    (df_holidays["is_national"] == 1) & (df_holidays["type"] == "Holiday")
]["date"].unique()

df_train_temp = df_train.copy()
df_train_temp["is_natl_holiday"] = df_train_temp["date"].isin(national_holidays).astype(int)

holiday_daily = df_train_temp.groupby(["date", "is_natl_holiday"])["sales"].sum().reset_index()
holiday_avg = holiday_daily.groupby("is_natl_holiday")["sales"].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 柱状图对比
labels = ["Non-Holiday", "National Holiday"]
colors = ["#5DADE2", "#E74C3C"]
bars = axes[0].bar(labels, holiday_avg.values, color=colors, edgecolor="white", width=0.4)
for bar, val in zip(bars, holiday_avg.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5000,
                 f'{val:,.0f}', ha='center', fontsize=13, fontweight='bold')
axes[0].set_title("Average Daily Sales: Holiday vs Non-Holiday", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Average Daily Sales")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K'))

# 节假日前后各 3 天的销量趋势
holiday_dates_list = sorted(national_holidays)
window_data = []
for hd in holiday_dates_list[:20]:  # 取前20个节日
    for offset in range(-7, 8):
        d = hd + pd.Timedelta(days=offset)
        day_sales = df_train_temp[df_train_temp["date"] == d]["sales"].sum()
        window_data.append({"offset": offset, "sales": day_sales})

window_df = pd.DataFrame(window_data).groupby("offset")["sales"].mean().reset_index()
axes[1].plot(window_df["offset"], window_df["sales"],
             marker='o', linewidth=2, color="#2C3E50", markersize=8)
axes[1].axvline(0, color="red", linestyle="--", linewidth=1.5, alpha=0.7)
axes[1].set_title("Sales Around Holidays (±7 Days)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Days from Holiday")
axes[1].set_ylabel("Avg Daily Sales")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K'))

annotate = axes[1].annotate("Holiday", xy=(0, window_df["sales"].max()),
            fontsize=9, color="red", ha='center')

plt.tight_layout()
plt.show()

In [ ]:
# Cell 9: 【图表7】促销对销量的影响
df_train_temp = df_train.copy()
df_train_temp["has_promo"] = (df_train_temp["onpromotion"] > 0).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 有促销 vs 无促销的销量分布
promo_sales = [
    df_train_temp[df_train_temp["has_promo"] == 0]["sales"],
    df_train_temp[df_train_temp["has_promo"] == 1]["sales"],
]
bp = axes[0].boxplot(promo_sales, labels=["No Promotion", "With Promotion"],
                      patch_artist=True, widths=0.4)
bp['boxes'][0].set_facecolor("#5DADE2")
bp['boxes'][1].set_facecolor("#F39C12")
axes[0].set_title("Sales Distribution: Promotion vs No Promotion", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Sales")
axes[0].set_ylim(0, df_train_temp["sales"].quantile(0.95))

# 按品类的促销效果
family_promo = df_train_temp.groupby("family").agg(
    no_promo_avg=("sales", lambda x: x[df_train_temp.loc[x.index, "has_promo"] == 0].mean()),
    promo_avg=("sales", lambda x: x[df_train_temp.loc[x.index, "has_promo"] == 1].mean()),
).reset_index()
family_promo["lift"] = family_promo["promo_avg"] / family_promo["no_promo_avg"] - 1
family_promo = family_promo.sort_values("lift")

colors_lift = ["#27AE60" if x > 0 else "#E74C3C" for x in family_promo["lift"]]
axes[1].barh(family_promo["family"], family_promo["lift"] * 100, color=colors_lift)
axes[1].axvline(0, color="black", linewidth=0.8)
axes[1].set_title("Promotion Lift by Family (%)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Sales Lift (%)")

plt.tight_layout()
plt.show()

In [ ]:
# Cell 10: Oil Price vs Sales — Dual-Axis Time Series
daily_sales_all = df_train.groupby("date")["sales"].sum().reset_index()
daily_sales_all["sales_ma30"] = daily_sales_all["sales"].rolling(30, center=True).mean()

# Merge oil prices
merged_oil_sales = daily_sales_all.merge(df_oil[["date", "dcoilwtico"]], on="date", how="inner")

fig, ax1 = plt.subplots(figsize=(16, 5))

# Sales (left axis)
line1 = ax1.plot(merged_oil_sales["date"], merged_oil_sales["sales_ma30"],
                 linewidth=1.5, color="#2E86C1", alpha=0.8, label="Sales (30-day MA)")
ax1.set_ylabel("Daily Total Sales (30-day MA)", color="#2E86C1")
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K'))
ax1.tick_params(axis='y', labelcolor="#2E86C1")

# Oil price (right axis)
ax2 = ax1.twinx()
line2 = ax2.plot(merged_oil_sales["date"], merged_oil_sales["dcoilwtico"],
                 linewidth=1, color="#E74C3C", alpha=0.5, label="Oil Price (WTI)")
ax2.set_ylabel("Oil Price (WTI)", color="#E74C3C")
ax2.tick_params(axis='y', labelcolor="#E74C3C")

# Legend
lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc="upper left")
ax1.set_title("Sales vs Oil Price (2013-2017)", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()

# Correlation
corr_val = merged_oil_sales["sales_ma30"].corr(merged_oil_sales["dcoilwtico"])
print(f"Sales (30-day MA) vs Oil Price correlation: {corr_val:.4f}")

In [ ]:
# Cell 11: Missing Value Visualization — Before Cleaning (raw data)
import matplotlib.patches as mpatches

# Reload raw data to show missing values before cleaning
df_train_raw = pd.read_csv(os.path.join(RAW_DIR, "train.csv"))
df_oil_raw = pd.read_csv(os.path.join(RAW_DIR, "oil.csv"))
df_trans_raw = pd.read_csv(os.path.join(RAW_DIR, "transactions.csv"))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# train.csv — missing value percentage per column
train_missing = df_train_raw.isnull().mean() * 100
if train_missing.sum() == 0:
    axes[0].text(0.5, 0.5, "No Missing Values ✅", ha='center', va='center',
                 fontsize=16, transform=axes[0].transAxes)
    axes[0].set_title("train.csv — Missing Values", fontsize=12, fontweight="bold")
else:
    axes[0].bar(train_missing.index, train_missing.values)
    axes[0].set_title("train.csv — Missing Values (%)", fontsize=12, fontweight="bold")

# oil.csv — missing values over time
df_oil_raw["date"] = pd.to_datetime(df_oil_raw["date"])
df_oil_raw["is_missing"] = df_oil_raw["dcoilwtico"].isnull().astype(int)
axes[1].scatter(df_oil_raw["date"], df_oil_raw["is_missing"],
                c=df_oil_raw["is_missing"].map({0: "#27AE60", 1: "#E74C3C"}),
                s=5, alpha=0.8)
axes[1].set_title(f"oil.csv — Missing Values ({df_oil_raw['is_missing'].sum()} / {len(df_oil_raw)})",
                   fontsize=12, fontweight="bold")
axes[1].set_yticks([0, 1])
axes[1].set_yticklabels(["Present", "Missing"])
axes[1].set_xlabel("Date")

# transactions — coverage (date × store combinations)
all_dates = pd.date_range(df_trans_raw["date"].min(), df_trans_raw["date"].max())
all_stores = df_trans_raw["store_nbr"].unique()
total_combos = len(all_dates) * len(all_stores)
present_combos = len(df_trans_raw)
missing_rate = (1 - present_combos / total_combos) * 100

axes[2].bar(["Present", "Missing (est.)"],
            [present_combos, total_combos - present_combos],
            color=["#27AE60", "#E74C3C"])
axes[2].set_title(f"transactions.csv — Coverage\n{present_combos:,} / {total_combos:,} ({missing_rate:.1f}% missing)",
                   fontsize=12, fontweight="bold")
axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))

plt.tight_layout()
plt.show()

In [ ]:
# Cell 12: 【图表10】各地区销量分布
city_sales = df_train.groupby("city")["sales"].sum().sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# 按城市的总销量
colors_city = sns.color_palette("crest", len(city_sales))
axes[0].barh(city_sales.index, city_sales.values / 1e6, color=colors_city, edgecolor="white")
axes[0].set_title("Total Sales by City", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Total Sales (Millions)")

# 按州的销量（饼图）
state_sales = df_train.groupby("state")["sales"].sum().sort_values(ascending=False)
wedges, texts, autotexts = axes[1].pie(
    state_sales.values, labels=state_sales.index,
    autopct='%1.1f%%', startangle=140,
    colors=sns.color_palette("Set2", len(state_sales)),
    pctdistance=0.8
)
for at in autotexts:
    at.set_fontsize(8)
axes[1].set_title("Sales Share by State", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.show()

---
## EDA Key Findings

1. **Scale**: ~3M training rows, 54 stores × 33 families × 5 years (2013–2017)
2. **Sales Distribution**: Heavily right-skewed; 31.3% of records are zero sales (many products don't sell on certain days); median = 11, mean = 358
3. **Time Trend**: Sales grew year-over-year, roughly doubling from 2013→2016; clear monthly seasonality and annual trend
4. **Day-of-Week Pattern**: Weekends have highest sales (especially Saturday); midweek is relatively lower
5. **Store Types**: Type A/B are large stores (high sales volume), Type D/E are smaller; distribution varies by city
6. **Promotion Effect**: Average sales with promo ≈1,138 vs ~158 without — roughly 7× lift. Varies significantly by product family
7. **Holiday Impact**: National holidays show dramatic sales changes (Christmas, New Year's spike)
8. **Oil Price**: Oil & sales show moderate negative correlation. Oil prices crashed in late 2014
9. **Special Events**: 2016-04-16 Manabi earthquake had visible impact on local sales
10. **Transactions**: Highly correlated with sales (r=0.84) — a very strong feature candidate

### Feature Engineering Priorities
- Lag features (7/14/28 days) + rolling statistics are critical
- Holidays need precise matching by store city/state (National/Regional/Local)
- Oil prices should be lagged (consumer response has delay)
- onpromotion lag effects (purchase pull-forward) must be considered
- Time-series models MUST use time-based validation split, NEVER random split